# Faceless video: animate scene images with LTX-Video (free GPU)
Kaggle: Settings > Accelerator = GPU T4 x2, Internet = On. Add your `scene_00.png ...` as a Dataset (Add Input). Run all, then download `clips/scene_XX.mp4` from the Output panel.
Colab: upload the PNGs to `/content`, GPU runtime, run all.
Start with `SCENES = [0]` to check quality before animating the rest.

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
import glob, os, torch
from PIL import Image
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

SCENES = [0]            # which scenes to animate; start with one: [0]. Later: list(range(7))
PORTRAIT = False        # True for 9:16, False for 16:9
PROMPTS = {
 "0": "The stickman hiker looks around nervously, head turning slightly left and right, chest rising as he breathes, soft sunset light. Static camera, flat 2D cartoon style, keep the same drawing.",
 "1": "The stickman sits on the log and raises his hand in a calm stop gesture, then lowers it, small nod of the head. Static camera, flat 2D cartoon style.",
 "2": "The white owl on the branch blinks and tilts its head while the stickman hiker looks up at it, small body movement. Static camera, flat 2D cartoon style.",
 "3": "The stickman hiker leans a branch against the log with his arm, adjusting it, head nodding slightly. Static camera, flat 2D cartoon style.",
 "4": "The stickman hiker blows the whistle, cheeks puffing, sound waves pulsing, the owl in the background tilts its head. Static camera, flat 2D cartoon style.",
 "5": "The campfire flames flicker, stars twinkle, the stickman and the owl sway gently and look at the fire. Static camera, flat 2D cartoon style.",
 "6": "The stickman waves happily while the rescuers move their flashlights and wave back, light glowing at dawn. Static camera, flat 2D cartoon style."
}

# find the scene images wherever they were uploaded (Kaggle dataset, Colab files, current folder)
cands = glob.glob("/kaggle/input/**/scene_*.png", recursive=True) + glob.glob("/content/**/scene_*.png", recursive=True) + glob.glob("./**/scene_*.png", recursive=True)
imgs = {int(os.path.basename(p)[6:8]): p for p in cands}
print("images found:", sorted(imgs))

pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

W, H = (384, 704) if PORTRAIT else (704, 384)
os.makedirs("clips", exist_ok=True)
for i in SCENES:
    img = Image.open(imgs[i]).convert("RGB")
    # center-crop to the target aspect, then resize
    tw, th = (W, H)
    s = max(tw / img.width, th / img.height)
    img = img.resize((round(img.width * s), round(img.height * s)))
    l, t = (img.width - tw) // 2, (img.height - th) // 2
    img = img.crop((l, t, l + tw, t + th))
    out = pipe(image=img, prompt=PROMPTS[i], negative_prompt="worst quality, blurry, jittery, distorted, deformed, morphing, extra limbs, text",
               width=W, height=H, num_frames=97, num_inference_steps=30,
               generator=torch.Generator().manual_seed(42))
    export_to_video(out.frames[0], f"clips/scene_{i:02d}.mp4", fps=24)
    print("done", i)


In [ ]:
!ls -la clips